## Universidad Autonoma de Aguascalientes
## Departamento: Ciencias de la computacion
## Carrera: Ingenieria en Computacion Inteligente
## Curso: Machine y Deep Learning
## Maestro: Dr. Francisco Javier Luna Rosas
## Alumno: Guillermo González Lara (237864)
## Semestre: Enero_Junio del 2026

---

# PRÁCTICA No. 26: CNN(Mnist)

### La CNN se implementara para predecir imagenes de la base de datos Mnist (10 numeros escritos a mano) 
### La base de datos es un conjunto de datos muy utilziado en el aprendizaje automatico y la vision por computadora. Contene imagenes de digitos escritos a mano del 0 al 9, con las siguienes caracteristicas: 70,000 imagenes en total 60,000 para entrenamiento y 10,000 para prueba, imagenes en escala de grises de 28x28 pixeles, y etiquetadas con el  numero correspondiente




## Paso 1: Importar las librerías necesarias

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# 1. Definir rutas y cargar los datasets
train_dir = 'train' # Ruta a tu carpeta train
test_dir = 'test'   # Ruta a tu carpeta test

# Keras inferirá las clases automáticamente desde las subcarpetas
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    color_mode='grayscale',
    image_size=(48, 48), # Tamaño estándar de FER-2013
    batch_size=64
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    color_mode='grayscale',
    image_size=(48, 48),
    batch_size=64
)

# 2. Normalizar los valores de los píxeles (de 0-255 a 0-1)
# Esto ayuda a que la red neuronal aprenda más rápido
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# 3. Construcción de la CNN (Arquitectura)
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Apaga el 50% de las neuronas para evitar sobreajuste (overfitting)
    layers.Dense(7, activation='softmax') # 7 neuronas de salida (una por emoción)
])

# 4. Compilar el modelo
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 5. Entrenar el modelo
print("Iniciando entrenamiento...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25 # Puedes ajustar este número
)

# 6. Guardar el modelo entrenado
model.save('modelo_emociones.h5')
print("Modelo guardado como 'modelo_emociones.h5'")

### Fase 2: Ejecución en Tiempo Real (demo_camara.py)

Una vez que ejecutes el código anterior y se genere tu archivo modelo_emociones.h5, puedes usar este segundo script para encender tu cámara web y ver las predicciones en vivo.

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Cargar el modelo entrenado
model = load_model('modelo_emociones.h5')

# 2. Definir el diccionario de emociones
# IMPORTANTE: El orden debe coincidir con el orden alfabético de tus carpetas
emociones = ['Enojado', 'Disgusto', 'Miedo', 'Feliz', 'Neutral', 'Triste', 'Sorpresa']

# 3. Cargar el detector de rostros de OpenCV
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 4. Iniciar la captura de video (0 suele ser la cámara web integrada)
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Voltear la imagen tipo espejo (opcional, suele ser más cómodo visualmente)
    frame = cv2.flip(frame, 1)
        
    # Convertir a escala de grises
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detectar rostros
    rostros = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in rostros:
        # Dibujar cuadro alrededor del rostro
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        
        # Recortar solo el rostro (Región de Interés - ROI)
        roi_gray = gray[y:y+h, x:x+w]
        
        # Redimensionar al tamaño esperado por el modelo (48x48)
        try:
            roi_gray = cv2.resize(roi_gray, (48, 48))
            
            # Preparar la imagen para Keras: (1, 48, 48, 1) y normalizar
            img_array = roi_gray.astype('float32') / 255.0
            img_array = np.expand_dims(img_array, axis=0)
            img_array = np.expand_dims(img_array, axis=-1)
            
            # Predecir
            prediccion = model.predict(img_array, verbose=0)
            indice_emocion = np.argmax(prediccion)
            emocion_detectada = emociones[indice_emocion]
            
            # Mostrar texto en pantalla
            cv2.putText(frame, emocion_detectada, (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
        except Exception as e:
            # En caso de que el recorte falle por salirse de los márgenes
            print("Error al redimensionar:", e)

    # Mostrar ventana
    cv2.imshow('Deteccion de Emociones', frame)

    # Salir al presionar 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()